# Biped Balance Policy Training (mjlab + RSL-RL PPO on Colab)

This notebook trains the biped's balance policy using **mjlab** (MuJoCo Warp) and **RSL-RL**'s PPO on a Colab GPU runtime.

**Requires a GPU runtime**: Runtime > Change runtime type > GPU.

This notebook expects the upload bundle described in `docs/colab_upload_manifest.md` (created in a later sub-task) to already be present in the working directory before running the cells below.

In [ ]:
!nvidia-smi

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected - set Runtime > Change runtime type > GPU"
print(f"CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0)}")

## Weights & Biases logging (optional)

RSL-RL logs training metrics to **Weights & Biases** (W&B) by default. You can run training offline or log in with an API key to sync results to the cloud. Choose one of the options below, or skip both if you want to disable W&B entirely.

### Use W&B offline (optional)

Disable W&B cloud sync if you don't want to log in.

In [ ]:
!wandb offline

### Or login using an API key from your W&B account (optional)

Enable cloud logging if you want to track results on the W&B dashboard.

In [ ]:
!wandb login

## Mount Google Drive (required — checkpoints must persist here)

**Do this before training.** The Colab runtime's local disk is ephemeral — it is wiped
whenever the runtime disconnects or recycles, which will happen on any long training run.
Without Drive, a multi-hour run can finish (or get interrupted) and leave you with nothing
to show for it. Mounting Drive and pointing `--log-root` at it (done in the Train cell below)
means every checkpoint RSL-RL saves lands in your Drive immediately, not just at the end.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Single place to change the checkpoint location. Everything under this path is written
# to your actual Google Drive (via the mount above), so it survives runtime disconnects.
# notebooks/view_biped.ipynb defaults to this same path when looking for checkpoints to
# replay -- keep them in sync if you change this.
DRIVE_LOG_ROOT = "/content/drive/MyDrive/biped_training/logs/rsl_rl"
os.makedirs(DRIVE_LOG_ROOT, exist_ok=True)
print("Checkpoints will be saved under:", DRIVE_LOG_ROOT)


In [ ]:
# The bundle's requirements-colab.txt must already be present in the
# working directory (see the upload-bundle cell below).
!pip install -r requirements-colab.txt

In [ ]:
# Verify the upload bundle matches docs/colab_upload_manifest.md before
# continuing. The manifest documents the exact minimal set of files this
# notebook expects in the working directory (biped_warp.xml, meshes/,
# mjlab_biped/, requirements-colab.txt) - see that file for the full
# layout and rationale.
import os

expected_paths = [
    "models/mjcf/biped_warp.xml",
    "meshes/stl",
    "mjlab_biped",
    "requirements-colab.txt",
]
missing = [p for p in expected_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Upload bundle is incomplete, missing: {missing}. "
        f"See docs/colab_upload_manifest.md for the full expected bundle."
    )
print("Bundle looks complete:", os.listdir("."))


In [ ]:
import mjlab
import mujoco_warp
import rsl_rl
import importlib.metadata


def _version(pkg_name, module):
    try:
        return module.__version__
    except AttributeError:
        return importlib.metadata.version(pkg_name)


print("mjlab version:", _version("mjlab", mjlab))
print("mujoco_warp version:", _version("mujoco-warp", mujoco_warp))
print("rsl_rl version:", _version("rsl-rl-lib", rsl_rl))

## Register the biped task and run a smoke test

`mjlab_biped/mjlab_task.py` is the first module in this bundle that imports
the *real* `mjlab` package -- it translates the already-tested pure-Python
specs in the rest of `mjlab_biped/` into real mjlab manager-API objects and
registers `"Mjlab-Biped-Balance-v0"`. Several attribute-name guesses in
that file are explicitly UNVERIFIED (see `docs/mjlab_adapter_notes.md`) --
this smoke test is designed to fail FAST and CHEAP on any of them, before
any real (multi-hour) training run starts.


In [ ]:
import mjlab_biped.mjlab_task as biped_task

print(f"Registered task: {biped_task.TASK_ID}")


In [ ]:
# Minimal smoke test: build a tiny (num_envs=1) real mjlab env, reset it,
# step it once with a zero action, and print observation shapes. This is
# expected to catch any remaining API mismatches in mjlab_task.py
# immediately (AttributeError/TypeError) rather than deep into a real
# training run. See docs/mjlab_adapter_notes.md for what to check first
# if this cell raises an error -- most of the earlier UNVERIFIED items
# were already resolved and fixed via local CPU testing on 2026-07-11
# (see that doc's changelog), so a NEW failure here likely means
# something changed in mjlab itself since then.
#
# Expected output as of 2026-07-11: actor obs shape (1, 140) -- 28 dims
# per frame (6 actuated + 2 passive joint pos, same for vel, + 3 gyro +
# 6 previous_action + 3 velocity_command), stacked over a 5-frame history
# via ObservationGroupCfg(history_length=5); critic obs shape (1, 43),
# single frame, no stacking.
import torch

smoke_env_cfg = biped_task.make_biped_env_cfg(num_envs=1)
from mjlab.envs import ManagerBasedRlEnv

# device= is a required positional/keyword arg on real mjlab's
# ManagerBasedRlEnv (confirmed 2026-07-11 -- omitting it raises
# `TypeError: missing 1 required positional argument: 'device'`).
# Use "cuda" here since this notebook already asserted a GPU runtime
# above; the same smoke test also runs on "cpu" locally on a Mac with
# no GPU (see docs/mjlab_adapter_notes.md's "Local CPU testing" section)
# for validating this file before spending a Colab session.
env = ManagerBasedRlEnv(cfg=smoke_env_cfg, device="cuda")
obs, extras = env.reset()
print("actor obs shape:", obs["actor"].shape)
print("critic obs shape:", obs["critic"].shape)
assert obs["actor"].shape == (1, 140), f"expected actor shape (1, 140), got {obs['actor'].shape}"
assert obs["critic"].shape == (1, 43), f"expected critic shape (1, 43), got {obs['critic'].shape}"
assert torch.isfinite(obs["actor"]).all(), "actor obs contains NaN/Inf"
assert torch.isfinite(obs["critic"]).all(), "critic obs contains NaN/Inf"

zero_action = torch.zeros(1, 6, device=obs["actor"].device)
obs, reward, terminated, truncated, extras = env.step(zero_action)
print("post-step reward:", reward)
print("post-step terminated:", terminated.item())
print("post-step actor obs finite:", torch.isfinite(obs["actor"]).all().item())
assert not terminated.item(), (
    "episode terminated after a single zero-action step -- this exact "
    "symptom means the stand-pose reset isn't being applied (see "
    "docs/mjlab_adapter_notes.md's 'events={}' bug writeup); check "
    "make_biped_env_cfg()'s events= wiring before anything else."
)

print("\nSmoke test passed. Safe to proceed to training below.")


## Train

**Use `scripts/colab_train.py`, not the bare `train` console script.**
mjlab's `train`/`play` CLIs only auto-register external tasks through a
`"mjlab.tasks"` Python entry-point group (see
`docs/mjlab_adapter_notes.md`'s "train/play CLI task discovery" section) --
that mechanism requires `mjlab_biped` to be a properly pip-installed
package, which this bundle deliberately is not (it's just an unzipped
directory). `scripts/colab_train.py` sidesteps this by importing
`mjlab_biped.mjlab_task` (which registers the task as a side effect,
exactly as the earlier registration cell already does) in the same
process before calling mjlab's real training entry point -- same CLI
flags as the `train` console script itself, just invoked via `python`.

Start with a small `num_envs` and `max_iterations` for a real (not just
smoke-test) training check before scaling up -- e.g. a few hundred
iterations to see the reward trend, before committing to a long run.

**Flags worth knowing about (all real mjlab `TrainConfig`/`RslRlBaseRunnerCfg`
fields, verified against the installed CLI -- not guessed):**
- `--log-root {DRIVE_LOG_ROOT}` -- writes checkpoints straight to the Drive path
  mounted above instead of the ephemeral local disk. This is why the Drive-mount
  section above must run first.
- `--agent.experiment-name <name>` -- groups runs under
  `{log_root}/{experiment_name}/<timestamp>/model_*.pt`. Defaults to mjlab's own
  generic `"exp1"` if you don't set it; pick something identifiable if you'll run
  more than one experiment.
- `--agent.wandb-project <name>` -- **by default this is `"mjlab"`**, mjlab's own
  generic project name (see `mjlab/rl/config.py`'s `RslRlBaseRunnerCfg.wandb_project`
  default) -- every mjlab task anyone runs with the default settings logs to that
  same project. Set this to something like `biped-balance` so your runs land in
  their own W&B project instead of mixing with the generic default.
- `--agent.run-name <label>` -- optional extra label appended to the timestamped
  run directory and shown as the W&B run's display name.


In [ ]:
!python scripts/colab_train.py Mjlab-Biped-Balance-v0 \
    --env.scene.num-envs 4096 --agent.max-iterations 300 \
    --log-root {DRIVE_LOG_ROOT} \
    --agent.wandb-project biped-balance \
    --agent.experiment-name biped-run1


## Eval (`play`)

Renders rollouts from a saved checkpoint. **Uses `scripts/colab_play.py`, not
the bare `play` console script** -- same reason as the train cell above.

Checkpoints now live under `DRIVE_LOG_ROOT` (on Drive), not the local disk, so
they persist across runtime restarts too -- you can come back later, re-mount
Drive, and re-run just this section without re-training. The cell below
auto-picks the newest checkpoint under `DRIVE_LOG_ROOT`; set `checkpoint_path`
explicitly if you want a specific one instead.


In [ ]:
import glob

checkpoint_path = ""  # leave empty to auto-pick the newest checkpoint below
if not checkpoint_path:
    ckpts = glob.glob(os.path.join(DRIVE_LOG_ROOT, "**", "model_*.pt"), recursive=True)
    assert ckpts, f"No checkpoints found under {DRIVE_LOG_ROOT} -- train first, or set checkpoint_path explicitly."
    checkpoint_path = max(ckpts, key=os.path.getmtime)
    print("Auto-selected newest checkpoint:", checkpoint_path)

assert checkpoint_path, "Set checkpoint_path to a real checkpoint file before running this cell."
!python scripts/colab_play.py Mjlab-Biped-Balance-v0 --checkpoint-file {checkpoint_path}
